In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.1.0',
    'torchaudio>=2.1.0',
    'demucs>=4.0.0',
    'openai-whisper>=20231117',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyloudnorm>=0.1.1',
    'pyarrow>=16.0.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '*.whl'], check=True)
except Exception:
    pass

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')

from shared.secrets import load_secrets
from shared.cf_client import CFClient
from shared.workflow_kernel import WorkflowKernel
from shared.repo_router import RepoRouter
from shared.audio_utils import AudioUtils

import yaml
from pathlib import Path

CONFIG_DIR = Path('/kaggle/input/S2S-pipline-v2-0-2/config')
WORK_DIR   = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID       = 'run_20260507_001'
SESSION_ID   = 'gpu_clean_01'
SESSION_TYPE = 'gpu_clean'
SHARD_KEY    = 't4'
GPU_TYPE     = 'T4'
VRAM_LIMIT   = 16.0

SECRETS = load_secrets(require_gemini=False)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO   = repos_cfg['repos']['stage0_codec']['repo_id']
OVERFLOW_REPO = repos_cfg['repos']['overflow']['repo_id']

kernel = WorkflowKernel(
    run_id           = RUN_ID,
    session_id       = SESSION_ID,
    session_type     = SESSION_TYPE,
    shard_key        = SHARD_KEY,
    cf_worker_url    = SECRETS['CF_WORKER_URL'],
    cf_worker_secret = SECRETS['CF_WORKER_SECRET'],
    gpu_type         = GPU_TYPE,
    vram_limit_gb    = VRAM_LIMIT,
    session_max_hours = 8.5,
)
kernel.start()

# GPU health check
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'[gpu] {gpu_name} — {gpu_mem:.1f} GB VRAM')
    assert gpu_mem / 1e9 >= VRAM_LIMIT * 0.8, f'VRAM too low: {gpu_mem/1e9:.1f} < {VRAM_LIMIT*0.8:.1f}'
else:
    raise RuntimeError('[gpu] CUDA not available — T4 required for this session')

print(f'[session] {SESSION_ID} started — run={RUN_ID}, gpu={GPU_TYPE}')

In [ ]:
repo_router = RepoRouter(STAGE0_REPO, OVERFLOW_REPO)

stages_to_run = ['p1c', 'p1d']

for stage in stages_to_run:
    if kernel.session_expiring:
        print(f'[session] expiring — skipping {stage}')
        break
    kernel.check_session_time()

    started_at = kernel.log_stage_start(stage)
    try:
        if stage == 'p1c':
            # CPU-side audio cleaning: VAD, SNR, loudness normalization
            exec(open('/kaggle/input/S2S-pipline-v2-0-2/pipeline_1_collect/p1c_clean_cpu.ipynb').read())
        elif stage == 'p1d':
            # GPU-side: Demucs separation, Whisper transcription, language validation
            exec(open('/kaggle/input/S2S-pipline-v2-0-2/pipeline_1_collect/p1d_clean_gpu.ipynb').read())
        kernel.log_stage_end(stage, started_at)
        print(f'[session] {stage} completed')
    except Exception as e:
        kernel.log_stage_end(stage, started_at, error=str(e))
        print(f'[session] {stage} failed: {e}')
        raise

kernel.stop()
print('[session] gpu_clean session complete')